# 03 — Recorded mode: $0 and falsifiable offline

**Goal:** understand one-code-path/two-modes, and generate deterministic example responses from a schema — the trick that makes an integration testable before any live call.

**The principle (this whole course in one sentence):** the first deliverable of any wire integration is a free local simulation that can falsify it offline; live is the final check, never the debugger.

In [ ]:
# Live-mode guard: cells that invoke the real gecko CLI run only when you
# opt in AND npx is available. Everything else in this notebook is offline.
#   export GECKO_COOKBOOK_LIVE=1   # to enable live cells
import os
import shutil

GECKO_LIVE = os.environ.get("GECKO_COOKBOOK_LIVE") == "1" and shutil.which("npx") is not None
FIXTURES = None
from pathlib import Path
here = Path.cwd()
while not (here / "pyproject.toml").exists():
    here = here.parent
FIXTURES = here / "cookbook" / "fixtures"
print(f"live mode: {GECKO_LIVE} — fixtures: {FIXTURES}")

## 1. One code path, two modes

```
agent -> tool -> prepare request -> [transport edge] -> real API   (live)
agent -> tool -> prepare request -> [transport edge] -> synthesized  (recorded)
```

Everything up to the edge is IDENTICAL. If your recorded call is wrong, your live call is wrong — that's what makes recorded mode *falsifying*, not just cheap.

## 2. Deterministic examples from a schema (the miniature)

In [ ]:
import yaml

spec = yaml.safe_load((FIXTURES / "petstore-mini.yaml").read_text())


def sample_from_schema(schema: dict, spec: dict, depth: int = 0):
    """Deterministic schema -> example. No randomness: same input, same output."""
    if depth > 5:
        return None
    if '$ref' in schema:
        name = schema['$ref'].rsplit('/', 1)[-1]
        return sample_from_schema(spec['components']['schemas'][name], spec, depth + 1)
    kind = schema.get('type')
    if kind == 'object':
        return {
            key: sample_from_schema(sub, spec, depth + 1)
            for key, sub in schema.get('properties', {}).items()
        }
    if kind == 'array':
        return [sample_from_schema(schema.get('items', {}), spec, depth + 1)]
    if 'enum' in schema:
        return schema['enum'][0]          # first enum value, always
    if kind == 'integer':
        return schema.get('minimum', 0)
    if kind == 'number':
        return 0.0
    if kind == 'string':
        return 'example'
    return None

pet_schema = {'$ref': '#/components/schemas/Pet'}
print(sample_from_schema(pet_schema, spec))
assert sample_from_schema(pet_schema, spec) == sample_from_schema(pet_schema, spec)
print('deterministic: same schema, same example, every time')

## 3. What determinism buys

- **Tests**: assert against the synthesized response; no network, no key, no flake.
- **Demos**: a classroom of 30 gets identical output.
- **Falsification**: if the tool builds the wrong request or misparses the response shape, recorded mode catches it — for free.

This repo runs the same play with `FakeLLM`: the deterministic stand-in is the default; the live thing is an opt-in at the edge.

## 4. Try it against a comprehended surface (live)

With a surface added (notebook 02) and served, a recorded call runs the full path — tool selection, request preparation, response handling — with the transport edge synthesizing from schema. Cost: nothing. Confidence: everything except the wire itself.

In [ ]:
if GECKO_LIVE:
    print("With the surface from notebook 02 added, run: npx @geckovision/gecko serve")
    print("then exercise a recorded call from your MCP client (notebook 04).")
else:
    print("SKIP (offline): the miniature above IS the lesson; live wiring happens in 04.")